### setting up an agent brain

In [39]:
from ultralytics import YOLO
from autogen_agentchat.agents import AssistantAgent
from autogen_ext.models.ollama import OllamaChatCompletionClient
from dotenv import load_dotenv
import os
from doc_string import prompt

load_dotenv()  # Load environment variables from .env file

True

In [40]:
ollama_model_client = OllamaChatCompletionClient(
   model="qwen2.5:3b"
   )

In [44]:
def yolo_tool(path: str) -> str:
   """
    Analyze an image using YOLOv8 object detection.

    This tool takes the path of an image, runs YOLOv8 object detection,
    identifies the objects present in the image, and returns the detected
    objects along with their confidence scores and bounding box information.

    Args:
        image_path (str): The file path of the image to analyze.

    Returns:
        str: A summary of the objects detected in the image, including
        their class names and confidence scores.
   """
   model = YOLO("yolov8n.pt") 
   result = model(source=path, show=True)  # predict on an image and show the results
   print(result[0].show()) 
   return "YOLO analysis complete"

In [45]:
yolo_agents = AssistantAgent(
    name="YOLO_Agent",
    model_client=ollama_model_client,
    system_message="You are a helpful assistant that can analyze images using YOLOv8 and provide insights based on the detected objects.",
    tools = [yolo_tool],
    description="This agent can analyze images using YOLOv8 and provide insights based on the detected objects."
) 

In [46]:
x = await yolo_agents.run(task="who are you")

print(x.messages[-1].content)

I am a tool designed to analyze images using YOLOv8 and provide insights based on the detected objects. How can I assist you today?


In [47]:
response = yolo_tool("L:\\agentic-yolo\images\img-1.webp")
print(response)

<>:1: SyntaxWarning: invalid escape sequence '\i'
<>:1: SyntaxWarning: invalid escape sequence '\i'
C:\Users\JILLA SRI VARDHAN\AppData\Local\Temp\ipykernel_3348\665482810.py:1: SyntaxWarning: invalid escape sequence '\i'
  response = yolo_tool("L:\\agentic-yolo\images\img-1.webp")



image 1/1 L:\agentic-yolo\images\img-1.webp: 448x640 5 persons, 1 bench, 3 chairs, 97.5ms
Speed: 4.6ms preprocess, 97.5ms inference, 1.3ms postprocess per image at shape (1, 3, 448, 640)
None
YOLO analysis complete


### teams

In [48]:
from autogen_agentchat.teams import RoundRobinGroupChat
from autogen_agentchat.conditions import TextMentionTermination 

In [50]:
team = RoundRobinGroupChat(
   participants=[yolo_agents],
   max_turns=3,
   termination_condition=TextMentionTermination("goodbye")
)